# W6 Results Evaluation (molecule, facts_50k + paper + predicates)

W6 = W5 + two-sided predicates. Evaluate baselines against brute-force GT.

**Recall**: for each query, count baseline rows whose `score ≤ worst_GT_score`. `recall = hits / K`. Compare scores only (not fact_id/pmid).

**Extra breakdowns**: recall & latency by similarity tier, by predicate bucket, by filtered cartesian size `|T1|·|T2|·facts_sel·paper_sel`.

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path('.')
GT_FILE = 'w6_queries_100_gt.json'
WORKLOAD_PATH = Path('..') / '..' / 'workload' / 'w6' / 'w6_queries_100.json'

def load_results(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def gt_answer_to_scores(answer):
    return [(row['fact_id'], row['score']) for row in answer]

def baseline_answer_to_scores(answer):
    # molecule baseline rows: [fact_id, paper_id, score_dist, join_dist]
    return [(row[0], row[2]) for row in answer]

def query_results_to_map(data, is_gt=False):
    fn = gt_answer_to_scores if is_gt else baseline_answer_to_scores
    return {r['query_id']: fn(r['answer']) for r in data['results']}

assert (RESULTS_DIR / GT_FILE).exists(), f'Ground truth not found: {GT_FILE}'
gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_by_qid = query_results_to_map(gt_data, is_gt=True)

# workload (for sel / bucket / cart per query)
with open(WORKLOAD_PATH) as f:
    workload = json.load(f)
query_meta = {q['query_id']: q for q in workload['queries']}

BASELINE_FILES = sorted(
    f.name for f in RESULTS_DIR.glob('*.json')
    if f.name != GT_FILE
)

print(f'Ground truth: {GT_FILE}, queries = {len(gt_by_qid)}')
print(f'Baselines ({len(BASELINE_FILES)}):')
for f in BASELINE_FILES:
    print(f'  {f}')

Ground truth: w6_queries_100_gt.json, queries = 93
Baselines (5):
  20260420_021826.json
  20260420_021941.json
  20260420_022041.json
  20260420_022149.json
  20260420_022257.json


In [2]:
def eval_one_baseline(baseline_by_qid, gt_by_qid):
    per_query, total_hits, total_denom = [], 0, 0
    for qid, answers in baseline_by_qid.items():
        if qid not in gt_by_qid:
            continue
        gt_answers = gt_by_qid[qid]
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(score for _, score in gt_answers)
        m = sum(1 for _, score in answers if score <= an + 1e-6)
        per_query.append({'query_id': qid, 'K': n, 'hits': m, 'recall': m / n})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
per_query_by_baseline = {}
for f in BASELINE_FILES:
    data = load_results(RESULTS_DIR / f)
    by_qid = query_results_to_map(data, is_gt=False)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    name = Path(f).stem
    n_queries = len(per_query)
    total_sec = data.get('total_elapsed_sec', None)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        'baseline':    name,
        'mean_recall': mean_recall,
        'n_queries':   len(per_query),
        'qps':         qps,
    })
    per_query_by_baseline[name] = per_query
    print(f'{name}: mean_recall = {mean_recall:.4f}, qps = {qps:.2f}')

20260420_021826: mean_recall = 0.9957, qps = 1.98
20260420_021941: mean_recall = 0.9941, qps = 1.99
20260420_022041: mean_recall = 0.9941, qps = 1.94
20260420_022149: mean_recall = 0.9941, qps = 1.99
20260420_022257: mean_recall = 0.9941, qps = 1.94


In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

       baseline  mean_recall  n_queries      qps
20260420_021826     0.995699         93 1.982909
20260420_021941     0.994086         93 1.990051
20260420_022041     0.994086         93 1.936876
20260420_022149     0.994086         93 1.990620
20260420_022257     0.994086         93 1.944544
